# ROC & Precision-Recall Curves — Traditional ML Baseline
Input: results/predictions_for_curves.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    roc_auc_score,
)
from pathlib import Path

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COLORS = {"rf": "#2196F3", "svm": "#FF5722", "xgboost": "#4CAF50"}

df = pd.read_csv(RESULTS_DIR / "predictions_for_curves.csv")
print(f"Rows: {len(df):,} | Models: {sorted(df['model'].unique())} | Subjects: {df['subject'].nunique()}")
print(f"Label balance:\n{df['y_true'].value_counts(normalize=True).round(3)}")

In [ ]:
# All experiments and subjects pooled per model.
# This is the standard way to present a multi-fold baseline in a thesis.

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Traditional ML Baseline — All Experiments Pooled", fontsize=13, fontweight="bold")

prevalence = df["y_true"].mean()

for model_name, group in df.groupby("model"):
    color = MODEL_COLORS.get(model_name)

    # ROC
    fpr, tpr, _ = roc_curve(group["y_true"], group["y_prob"])
    roc_auc = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, lw=2, color=color, label=f"{model_name.upper()}  AUC={roc_auc:.3f}")

    # PR
    prec, rec, _ = precision_recall_curve(group["y_true"], group["y_prob"])
    ap = average_precision_score(group["y_true"], group["y_prob"])
    ax_pr.plot(rec, prec, lw=2, color=color, label=f"{model_name.upper()}  AP={ap:.3f}")

# ROC — random baseline
ax_roc.plot([0, 1], [0, 1], "k--", lw=1, label="Random  AUC=0.500")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title("ROC Curve")
ax_roc.legend(loc="lower right", fontsize=9)
ax_roc.set_xlim([0, 1]); ax_roc.set_ylim([0, 1.02])

# PR — no-skill baseline at stress prevalence
ax_pr.axhline(prevalence, color="k", ls="--", lw=1, label=f"No-skill  AP={prevalence:.3f}")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("Precision-Recall Curve")
ax_pr.legend(loc="upper right", fontsize=9)
ax_pr.set_xlim([0, 1]); ax_pr.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(FIGURES_DIR / "baseline_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/figures/baseline_curves.png")

In [ ]:
rows = []
for (model_name, exp), group in df.groupby(["model", "experiment"]):
    try:
        auc_val = roc_auc_score(group["y_true"], group["y_prob"])
        ap_val  = average_precision_score(group["y_true"], group["y_prob"])
    except Exception:
        auc_val, ap_val = float("nan"), float("nan")
    rows.append({"model": model_name, "experiment": exp,
                 "roc_auc": round(auc_val, 4), "avg_precision": round(ap_val, 4)})

summary = (pd.DataFrame(rows)
             .sort_values(["model", "roc_auc"], ascending=[True, False]))

summary.to_csv(RESULTS_DIR / "curves_summary.csv", index=False)
print("Saved: results/curves_summary.csv\n")
print(summary.to_string(index=False))